# Capstone: can this run on the library laptop?

MichAl Academy, lesson 4.11.

A primary school is putting together a local history project. They want a
helper on the laptop in the library that answers questions about the village
from the parish records the class has typed up. It has to work with the wifi
off, on the laptop they already own, with nobody paying a subscription.

They have found an open-weight model and they want to know whether to use it.

**Your job is not to make the model score well.** It is to end up able to write
one short note to the head teacher, who is not technical, saying what you would
do and what would change your mind.

Work the tasks in order. Each ends in a check that tells you when you have it.
Answers are folded at the bottom, and opening one early spends the task.

Budget about ninety minutes.


In [ ]:
import copy
import statistics
import time
import warnings

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")
torch.set_num_threads(1)

NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME)
model.eval()

N_PARAMS = sum(p.numel() for p in model.parameters())
print(f"{NAME}\n{N_PARAMS:,} parameters")


def ask(m, question, passage=None, temperature=0.0, max_new_tokens=60, seed=None):
    """One question to the model, optionally with a passage in front of it."""
    content = question if passage is None else f"{passage}\n\nQuestion: {question}"
    ids = tok.apply_chat_template(
        [{"role": "user", "content": content}],
        add_generation_prompt=True, return_tensors="pt")
    if seed is not None:
        torch.manual_seed(seed)
    out = m.generate(
        ids, max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature if temperature > 0 else None,
        pad_token_id=tok.eos_token_id,
        attention_mask=torch.ones_like(ids))
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()


print(ask(model, "In one sentence, what is a parish record?"))


## The material

Everything below is about **Cottersley**, a village that does not exist. It was
written for this exercise.

That is the point of it, and it is the one design decision in this notebook you
should carry into your own work. The model cannot have read about Cottersley,
because there is nothing to read. So any confident answer it gives without the
passage in front of it is invented, and you do not need a fact-checker to know
that: you know it by construction.

Compare this to testing on a real village. A right answer might come from the
passage or from something the model absorbed in training, and you would have no
way to tell which.


In [ ]:
PASSAGE = """Cottersley is a village in the parish of Aldenmere. The parish
records begin in 1698, when the first rector, Thomas Bramble, took office. The
village mill was built in 1743 on the river Fenn and ground flour until 1911,
when it burned down. A school was opened in 1871 with 38 pupils and one teacher,
Miss Eleanor Pike, who taught there for 41 years. The railway reached Cottersley
in 1856 and the station closed in 1964. The population was 412 in the 1901
census and 1,180 in the 2021 census. The war memorial on the green lists 23
names from the First World War and 7 from the Second."""

QUESTIONS = [
    ("In what year were the Cottersley parish records begun?", "1698"),
    ("Who was the first rector of Cottersley?", "Thomas Bramble"),
    ("In what year was the Cottersley mill built?", "1743"),
    ("What river was the Cottersley mill built on?", "Fenn"),
    ("How many pupils did the Cottersley school open with?", "38"),
    ("Who was the first teacher at the Cottersley school?", "Eleanor Pike"),
    ("In what year did the railway reach Cottersley?", "1856"),
    ("What was the population of Cottersley in the 2021 census?", "1,180"),
    ("How many names from the First World War are on the Cottersley war memorial?", "23"),
    ("In what year did the Cottersley mill burn down?", "1911"),
]

print(f"{len(PASSAGE.split())} words in the passage, {len(QUESTIONS)} questions")


def scores(answers):
    """An answer counts if the expected string appears in it. Crude on purpose:
    it never marks a wrong answer right, which is the direction that matters."""
    return sum(exp.replace(",", "") in ans.replace(",", "")
               for (_, exp), ans in zip(QUESTIONS, answers))


## Task 1: how long does one answer take?

Before anything about quality. The laptop has one job at a time and a class has
thirty children in it.

Measure how long the model takes to answer one question, and how many tokens it
produced, then work out the rate.

Lesson 4.1.3 is the unit about tokens being the thing that gets counted.


In [ ]:
t0 = time.time()
sample_answer = ask(model, QUESTIONS[0][0], passage=PASSAGE, max_new_tokens=60)
elapsed = time.time() - t0

n_out = len(tok(sample_answer).input_ids)
print(f"answer   : {sample_answer!r}")
print(f"took     : {elapsed:.2f}s for {n_out} tokens")

tokens_per_second = None    # TODO: from the two numbers above
seconds_for_class = None    # TODO: one answer each for 30 children, one at a time

print()
print("tokens per second       :", tokens_per_second)
print("seconds for 30 children :", seconds_for_class)
print()
print("task 1 done?", tokens_per_second is not None and seconds_for_class is not None)


## Task 2: what does it say when it has not been told?

Ask all ten questions with **no passage**. Cottersley does not exist, so there is
no correct answer available to the model.

What you are counting is not right answers. It is how many replies sound like
answers: a date, a name, a number, stated without hedging.

Lesson 4.7.1 is the unit on why this happens.


In [ ]:
blind = [ask(model, q, passage=None) for q, _ in QUESTIONS]
for (q, _), a in zip(QUESTIONS, blind):
    print(f"Q {q}\nA {a[:120]}\n")

print("scored as correct by the marker:", scores(blind), "of", len(QUESTIONS))


In [ ]:
n_confident = None   # TODO: how many of the ten replies state a fact about
                     #       Cottersley rather than saying it does not know
n_refused   = None   # TODO: how many say, in some form, that they do not know

print("confident inventions :", n_confident)
print("admissions of not knowing :", n_refused)
print()
print("task 2 done?", n_confident is not None and n_refused is not None
      and n_confident + n_refused == len(QUESTIONS))


## Task 3: does the passage fix it?

Now ask the same ten questions with the passage in front of them, and score
against the answer key.

Nothing about the model changes here. The only difference is what is in the
context window.

Lesson 4.7.3 is the unit this task measures.


In [ ]:
with_passage = [ask(model, q, passage=PASSAGE) for q, _ in QUESTIONS]
for (q, exp), a in zip(QUESTIONS, with_passage):
    hit = exp.replace(",", "") in a.replace(",", "")
    print(f"{'OK ' if hit else '   '} {q}\n    expected {exp!r}\n    got      {a[:100]!r}\n")

open_book = scores(with_passage)
print(f"with the passage: {open_book} of {len(QUESTIONS)}")


In [ ]:
gain = None    # TODO: open_book minus the score from task 2's `blind` answers
verdict = ""   # TODO: one sentence. Is the remaining error the model failing to
               #       read, or the marker being too strict? Look at the misses.

print("gain from the passage:", gain)
print("verdict:", verdict)
print()
print("task 3 done?", gain is not None and verdict != "")


## Task 4: which temperature would you ship?

Run one question five times at temperature 0 and five times at temperature 1.0.

Then decide. This is a judgement, not a lookup: a helper that always says the
same thing is predictable and boring, and one that varies is interesting and
sometimes wrong. The school has to live with whichever you pick.

Lesson 4.5.1 is the setting and Lesson 4.5.4 is why repeats can differ even
without it.


In [ ]:
q, exp = QUESTIONS[2]
cold = [ask(model, q, passage=PASSAGE, temperature=0.0) for _ in range(5)]
warm = [ask(model, q, passage=PASSAGE, temperature=1.0, seed=s) for s in range(5)]

print(f"question: {q}  (expected {exp})")
print("\ntemperature 0")
for a in cold:
    print("  ", a[:90])
print("\ntemperature 1.0")
for a in warm:
    print("  ", a[:90])
print(f"\ndistinct answers: {len(set(cold))} cold, {len(set(warm))} warm")
print(f"correct: {sum(exp in a for a in cold)}/5 cold, {sum(exp in a for a in warm)}/5 warm")


In [ ]:
# No single right answer here. Write down the one you would ship and why.
ship_temperature = None   # TODO: a number
because          = ""     # TODO: one sentence naming what you are trading away

print("ship at temperature:", ship_temperature)
print("because            :", because)
print()
print("task 4 done?", ship_temperature is not None and because != "")


## Task 5: is the quantized version good enough?

The laptop is old. Quantizing the model makes it smaller and usually faster, and
Lesson 4.10.2 measured what it costs.

Do it, then re-run task 3's scoring on the quantized model and compare. The
decision at the end is yours, and there is no single right answer: a smaller,
faster helper that gets one more question wrong may or may not be the better
thing to put in a school library.


In [ ]:
import os
import tempfile


def disk_size(m):
    with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as f:
        torch.save(m.state_dict(), f.name)
        size = os.path.getsize(f.name)
    os.unlink(f.name)
    return size


qmodel = torch.quantization.quantize_dynamic(
    copy.deepcopy(model), {torch.nn.Linear}, dtype=torch.qint8)
qmodel.eval()

print(f"float32 : {disk_size(model) / 1024**2:7.1f} MiB")
print(f"int8    : {disk_size(qmodel) / 1024**2:7.1f} MiB")


In [ ]:
q_answers = None    # TODO: the same ten questions, with the passage, on qmodel
q_score   = None    # TODO: scores(q_answers)

print("quantized score:", q_score)


In [ ]:
would_you_ship_it = None   # TODO: True or False
what_it_costs     = ""     # TODO: one sentence saying what the school gives up

print("ship the quantized one?", would_you_ship_it)
print("what it costs          :", what_it_costs)
print()
print("task 5 done?", would_you_ship_it is not None and what_it_costs != "")


## Task 6: does it fit?

The laptop has 8 GiB of memory and the school would like two children using it
at once.

Work out what that needs. Lesson 4.10.5 has the three terms and
Lesson 4.5.3 has the cache arithmetic. The model's own config holds every
number you need, and the cell below prints the ones that matter.


In [ ]:
cfg = model.config
print(f"blocks (hidden layers) : {cfg.num_hidden_layers}")
print(f"key/value heads        : {cfg.num_key_value_heads}")
print(f"head dimension         : {cfg.hidden_size // cfg.num_attention_heads}")
print(f"context window         : {cfg.max_position_embeddings}")
print(f"parameters             : {N_PARAMS:,}")


In [ ]:
BYTES_PER_NUMBER = 2      # float16

weights_mib      = None   # TODO: parameters x bytes, in MiB
cache_per_token  = None   # TODO: bytes for one token, using the printed numbers
conversation_len = None   # TODO: tokens you actually need, not the window size.
                          #       Task 3 had a passage and a question in context.
total_mib        = None   # TODO: weights, plus cache for two children at once

print("weights            :", weights_mib, "MiB")
print("cache per token    :", cache_per_token, "bytes")
print("total for two users:", total_mib, "MiB")
print()
print("task 6 done?", total_mib is not None and total_mib < 8 * 1024)


## Task 7: the note

One paragraph to the head teacher. No numbers they cannot act on, no jargon.

It has to answer three things: whether to use it, what the children will see go
wrong, and what would change your answer.

The third is the one that matters. A recommendation with no stated conditions is
an opinion.


In [ ]:
note = """
TODO: replace this with your paragraph.
"""

words = len(note.split())
print(note)
print(f"{words} words")
print()
print("task 7 done?", words > 40 and "TODO" not in note)


## Answers

Open one only after you have finished that task. They are working notes, not a
mark scheme, and several of them have no single right answer by design.

<details>
<summary>Task 1: the rate</summary>

Divide the token count by the elapsed seconds. On one CPU thread this model
produces on the order of tens of tokens a second, and a sixty-token answer
therefore takes a few seconds.

Thirty children, one at a time, is thirty of those. The number itself matters
less than the shape of it: this is a queue, not a service. If two children are
waiting, the second one waits for the first.

The reason to do this first is that it can end the exercise. A helper that takes
a minute per answer does not get used, however good the answers are.

</details>

<details>
<summary>Task 2: what it says when it has not been told</summary>

**Ten confident inventions, no refusals, nothing correct.** The parish records
begin in 1850, the first rector is John Cottersley, the mill stands on the
Cottersley River, the school opens with 100 pupils in 1995, and the mill burns
down in 1999.

Every one of those is fabricated, and you know it without checking anything,
because there is no Cottersley to be right about.

The detail worth noticing is that not one reply said it did not know. Lesson
4.7.1 is why: nothing in training scored whether an answer was true, so a
plausible date costs the model nothing and an admission of ignorance is not
rewarded either.

The name invention is the sharpest of them. "John Cottersley" and "Mrs. Emily
Cotterley" are what a model produces when it has no fact and a strong prior about
what village-history answers look like.

</details>

<details>
<summary>Task 3: what the passage buys</summary>

**0 of 10 becomes 8 of 10**, and the model is identical. Only the context
changed. That is the whole argument for putting the records in front of it rather
than hoping it knows them.

The two misses have different causes, and telling them apart is the task.

**The school question is the model.** Asked how many pupils the school opened
with, it answers "412 pupils in 1901": a real number from the passage, attached
to the wrong question. The fact was in front of it and it took the wrong one.

**The teacher question is the harness.** It answers "the first teacher at the
Cottersley school was Miss Eleanor" and stops, because `max_new_tokens` is 60 and
the reply ran out of room. The model was right and the notebook cut it off.
Raising the limit fixes it, and nothing about the model needed to change.

Reporting those two as one number, "8 of 10", hides the difference between a
model problem and a settings problem.

**And look at the river answer, which scored as correct.** "The Cottersley mill
was built on the River Fenn, which is a tributary of the River Thames." The first
half is from the passage. The second half is invented, and the marker cannot see
it. Putting the facts in the context reduced the invention; it did not stop it.

</details>

<details>
<summary>Task 4: the temperature</summary>

Temperature 0 gives the same answer every time. Temperature 1 gives several.

There is no right answer here, but there is a wrong way to decide it: picking the
one that scored better on five samples of one question. Five is not enough to
separate two settings, and you already know that from Lesson 4.6.4.

The defensible argument for 0 is that a fact lookup for children should be
repeatable, and a teacher who checks an answer wants to see the same one. The
defensible argument against is that 0 is not actually deterministic in a served
system, for the reason in Lesson 4.5.4, so promising repeatability is a promise
you may not be able to keep.

</details>

<details>
<summary>Task 5: the quantized model</summary>

Smaller on disk, usually faster, and the score on ten questions either holds or
drops by one or two.

The trap is treating a difference of one question in ten as a measurement. It is
not: ten questions cannot separate two models that are close, and the honest
statement is that this test was too small to tell them apart.

What you can say is what the school gives up: an old laptop that runs the
quantized model and cannot run the other one has made the decision for you.

</details>

<details>
<summary>Task 6: the memory</summary>

Weights at float16 come to roughly 257 MiB for this model.

The cache is `2 x blocks x key/value heads x head dimension x bytes`, per token.
For the printed config that is a few tens of kilobytes a token.

The number most people get wrong is the conversation length. Using the full
context window prices something nobody is doing; the passage and a question are
a few hundred tokens. Using the length you actually need is the difference
between a comfortable fit and a panic.

Two children at once is two caches, not one.

</details>

<details>
<summary>Task 7: the note</summary>

There is no model answer. There is a test: could the head teacher act on it
without asking you a follow-up question?

Three things separate a good note from a bad one.

**It says what the children will see**, not what the model does. "It will
sometimes make up a date" is actionable. "It has a next-token objective with no
truth term" is not.

**It names the condition.** Something like: use it only for questions the typed-up
records actually answer, and not as a general helper, because the difference
between tasks 2 and 3 is the whole of its usefulness.

**It says what would change the answer.** More records, a faster laptop, or an
older class that can check what they are told are all real answers. "A better
model" on its own is not, because Lesson 4.8.3 is the reason a bigger model
does not add a step that checks whether something is true.

</details>
